# Practice with the Python ATProtoSDK
This ipython notebook will walk you through the basics of working with the
ATProto python sdk. The content here heavily draws on [these examples](https://github.com/MarshalX/atproto/tree/main/examples)

In [9]:
from atproto import Client
from dotenv import load_dotenv
import os
import pprint


load_dotenv(override=True)
USERNAME = os.getenv("USERNAME")
PW = os.getenv("PW")

## Logging into your account

In [10]:
client = Client()
profile = client.login(USERNAME, PW)
pprint.pprint(profile.__dict__)

{'associated': ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'),
 'avatar': 'https://cdn.bsky.app/img/avatar/plain/did:plc:e7c5jfnl5ghoye3koawmhjva/bafkreicz64743eqmgwhbs74h7wklelwvelvjkvuawdw3njirftcrumwvma@jpeg',
 'banner': None,
 'created_at': '2025-11-06T01:56:43.052Z',
 'description': None,
 'did': 'did:plc:e7c5jfnl5ghoye3koawmhjva',
 'display_name': '',
 'followers_count': 0,
 'follows_count': 1,
 'handle': 'tnsteam.bsky.social',
 'indexed_at': '2025-11-06T01:56:43.052Z',
 'joined_via_starter_pack': None,
 'labels': [],
 'pinned_post': None,
 'posts_count': 0,
 'pronouns': None,
 'py_type': 'app.bsky.actor.defs#profileViewDetailed',
 'status': None,
 'verification': None,
 'viewer': ViewerState(activity_subscription=None, blocked_by=False,

## Working with posts

In [11]:
def post_from_url(client: Client, url: str):
    """
    Retrieve a Bluesky post from its URL
    """
    parts = url.split("/")
    rkey = parts[-1]
    handle = parts[-3]
    return client.get_post(rkey, handle)

post = post_from_url(client, "https://bsky.app/profile/labeler-test.bsky.social/post/3lktj7ewxxv2q")
pprint.pprint(post.value.__dict__)

{'created_at': '2025-03-20T20:14:57.103160+00:00',
 'embed': Main(images=[Image(alt='dog', image=BlobRef(mime_type='image/jpeg', size=169278, ref=IpldLink(link='bafkreibahplioamouecglrcqnshcxzdrwawdtwl5h676d2l7k7xbbti3pa'), py_type='blob'), aspect_ratio=None, py_type='app.bsky.embed.images#image')], py_type='app.bsky.embed.images'),
 'entities': None,
 'facets': None,
 'labels': None,
 'langs': ['en'],
 'py_type': 'app.bsky.feed.post',
 'reply': None,
 'tags': None,
 'text': 'check out this dog!'}


In [12]:
# https://github.com/MarshalX/atproto/blob/main/examples/profile_posts.py
prof_feed = client.get_author_feed(actor="weratedogs.com")
for i, feed_view in enumerate(prof_feed.feed[:10]):
    print(f"Post {i}:", feed_view.post.record.text)

post = prof_feed.feed[0].post
likes_resp = client.get_likes(post.uri, post.cid, limit=10)
print("Likes:", [like.actor.handle for like in likes_resp.likes])

post_thread_resp = client.get_post_thread(post.uri)
print([rep.post.record.text for rep in post_thread_resp.thread.replies[:10]])

Post 0: These are all the dogs we sponsored in October ❤️‍🩹
Post 1: SO many good dogs
Post 2: We only rate dogs. This is spaghetti con salsiccia. Obviously has the noodles and the sausage, but is notably missing sauce. Please only send dogs. Thank you… 13/10 (IG: futrinka_tacsko)
Post 3: This is Finn. His human stopped scratching his back, so he had to take matters into his own hands. 13/10 (TT: anna.ve.w)
Post 4: This dog was spotted wearing shoes. This is not a drill. He hopes you like them. 13/10
Post 5: This is Rufus. His Halloween costume arrived late. Really wants to escargot trick-or-treating if you don’t mind turning on your porch light. 13/10 (IG: rufuscorg)
Post 6: Here are the Top 5 Dogs of October!
Post 7: This is Reggie. He was just told about daylight savings time. More like daylight starving time. 13/10 (TT: melandreggie)
Post 8: This is Herman. He is a firm believer in keeping a consistent morning routine. No one said anything about efficient. 13/10 #SeniorPupSaturday (

## Followers/following

How might you use this information to investigate/mitigate a harm?

In [13]:

follower_resp = client.get_followers("weratedogs.com", limit=10)
following_resp = client.get_follows("weratedogs.com", limit=10)
print("Followers:", [follower.handle for follower in follower_resp.followers])
print("Following:", [follow.handle for follow in following_resp.follows])



Followers: ['jsnider2.bsky.social', 'kathleen3233.bsky.social', 'xjessicalane.bsky.social', 'schwartzhenryreal.bsky.social', 'gmfarfol.bsky.social', 'thegabster.bsky.social', 'waynehorton.bsky.social', 'artgal01.bsky.social', 'chesterdagger.bsky.social']
Following: ['15outof10.org']


## Exercise: Compute average dog ratings
The WeRateDogs account includes ratings out of 10 within some of its posts.
Write a script that computes the average rating (out of 10) for the 100 most
recent posts from this account. (note that not every post will have a rating)

In [ ]:
import re

def compute_avg_dog_rating(num_posts: int) -> float:
    """Return the average WeRateDogs rating (x/10) across the latest posts.

    Notes:
    - Scans only original posts (skips reposts without text).
    - Accepts integers or decimals (e.g., 12/10, 13.5/10).
    - Uses pagination until `num_posts` items are inspected or feed ends.
    """
    # Match numbers like 12/10 or 13.5/10, and avoid grabbing parts of larger numbers
    rating_re = re.compile(r'(?<!\d)(\d{1,2}(?:\.\d+)?)/10(?!\d)')

    ratings = []
    cursor = None
    seen = 0

    while seen < num_posts:
        batch_size = min(50, num_posts - seen)
        resp = client.get_author_feed(actor="weratedogs.com", limit=batch_size, cursor=cursor)
        feed_items = getattr(resp, "feed", [])
        if not feed_items:
            break
        for item in feed_items:
            record = getattr(item.post, "record", None)
            text = getattr(record, "text", None)
            if not isinstance(text, str) or not text:
                # Skip non-post records (e.g., pure reposts) or missing text
                seen += 1
                if seen >= num_posts:
                    break
                continue

            for m in rating_re.findall(text):
                try:
                    ratings.append(float(m))
                except ValueError:
                    pass  # ignore unexpected parse failures

            seen += 1
            if seen >= num_posts:
                break

        cursor = getattr(resp, "cursor", None)
        if not cursor:
            break

    return sum(ratings) / len(ratings) if ratings else 0.0

print("The average rating is:", compute_avg_dog_rating(100))








The average rating is: 13.0


## Exercise: Dog names
Collect the names of dogs within the latest 100 posts and print them to the
console. Hint: see if you can identify a pattern in the posts.

In [16]:
def collect_dog_names(num_posts):
    # TODO: complete
    return []

print("Here are the dog_names:", collect_dog_names(100))

Here are the dog_names: []


## Exercise: Soliciting donations
Some posts from the WeRateDogs account ask for donations -- usually for
covering medical costs for the featured dogs. Within the latest 100 posts, print
the text content of those that fall into this category

In [17]:
def donation_posts(num_posts):
    # TODO: complete
    return []

for post_text in donation_posts(100):
    print(post_text)